<a href="https://colab.research.google.com/github/hojuelita/text_classification/blob/main/Ejemplo_vectorizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de embedding

In [51]:
# Instalaciones
!pip install gensim
!pip install faiss-cpu
!pip install sentence_transformers

# Bibliotecas usadas
from collections import Counter
import numpy as np

# recuperamos el texto desde el git
import requests
import pandas as pd
import re # Added import for regular expressions

import gensim
from gensim.models import Word2Vec
import nltk
#import spacy

url_texto = "https://raw.githubusercontent.com/hojuelita/text_classification/refs/heads/main/data/Cuentos/A_una_bogotana.txt"
response = requests.get(url_texto)
texto = response.text

print("Original text:")

def clean_text(text, minusculas=True):
    # Remove everything inside brackets [...]
    text = re.sub(r"\[.*\]", '', text)
    # Remove everything inside parenthesis (...)
    text = re.sub(r'\(.*\)', '', text)
    # Remove continuous asterisks *
    text = re.sub(r'\*+', '', text)
    # Remove puncuaction signs
    text = re.sub(r'\á','a',text)
    text = re.sub(r'\é','e',text)
    text = re.sub(r'\í','i',text)
    text = re.sub(r'\ó','o',text)
    text = re.sub(r'\ú','u',text)
    text = re.sub(r'\,','',text)
    text = re.sub(r'\:','',text)
    text = re.sub(r'\;','',text)
    text = re.sub(r'\-',' ',text)
    text = re.sub(r'\.','',text)
    text = re.sub(r'\¡','',text)
    text = re.sub(r'\!','',text)
    text = re.sub(r'\¿','',text)
    text = re.sub(r'\?','',text)
    # Optional: remove leading/trailing whitespace that might result from cleaning
    if(minusculas):
      text = text.lower()
    text = text.strip()
    return text

def split_paragraphs(text):
    """
    Split the text into paragraphs based on double newlines.
    """
    paragraphs = text.split('\n\n')
    splitted = {}
    for j,par in enumerate(paragraphs):
      splitted[j] = re.sub("\n", "", par.strip())
    return splitted

cleaned_texto = clean_text(texto, minusculas=True)
print("\nCleaned text:")
print(cleaned_texto)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 32.5 MB/s  0:00:00
Original text:

Cleaned text:
a una bogotana 

el pasillo señora hermosa niña es como un lento y rosado vals vea
usted como aquellos dos enamorados pueden llevar el compas en medio de
la mas ardiente conversacion el dice que los lindos ojos de una mujer
valen por todos los astros y los lindos labios por todas las rosas
como ella quiere demostrar lo contrario le mira con los bellisimos ojos
suyos le sonrie con sus inefables labios que son en un todo iguales a
aquellos con que la señorita de abril dio el primer beso al caballero de
mayo el pasillo señora hermosa niña es como un lento y rosado vals

oh si si la fuerza de una pasion es mayor infinitas veces que el
empuje de ese enorme y poderoso tequendama usted conoce la catarata

dicen que sus aguas saltan de un clima a otro que alla abajo hay palmas
y flores que arriba en la roca que conocio la espada de bolivar hace
frio que delicia estar alla abajo señora dos 

In [43]:
# vamos a limpiar el texto quitando caraceres especiales,
# todo lo que esté entre []
parrafos = split_paragraphs(cleaned_texto)
parrafos


{0: 'a una bogotana',
 1: 'el pasillo señora hermosa niña es como un lento y rosado vals veausted como aquellos dos enamorados pueden llevar el compas en medio dela mas ardiente conversacion el dice que los lindos ojos de una mujervalen por todos los astros y los lindos labios por todas las rosascomo ella quiere demostrar lo contrario le mira con los bellisimos ojossuyos le sonrie con sus inefables labios que son en un todo iguales aaquellos con que la señorita de abril dio el primer beso al caballero demayo el pasillo señora hermosa niña es como un lento y rosado vals',
 2: 'oh si si la fuerza de una pasion es mayor infinitas veces que elempuje de ese enorme y poderoso tequendama usted conoce la catarata',
 3: 'dicen que sus aguas saltan de un clima a otro que alla abajo hay palmasy flores que arriba en la roca que conocio la espada de bolivar hacefrio que delicia estar alla abajo señora dos que se quieren lasoberana armonia de la naturaleza pondria un palio augusto y soberbio alidili

Una vez que tengamos los párrafos ya separados podemos vectorizarlos, lo vamos a hacer primero con word2vec pero también vamos a probarlo con FastText y Roberta

## Tokenization

In [44]:
# modelo para oraciones de nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# revisar el modelo usado por spacy

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [49]:
# Tokenize
oraciones = nltk.sent_tokenize(cleaned_texto, language='spanish')
oraciones_tokenizadas = [nltk.word_tokenize(sent, language='spanish') for sent in oraciones]

# Quitamos lo que no sean oraciones o palabras
oraciones_tokenizadas_limpias = [
    [re.sub(r'[^\w\s]','',token) for token in sent if len(token) > 1]
    for sent in oraciones_tokenizadas
]

print(len(f'Cantidad de oraciones {len(oraciones_tokenizadas_limpias)}'))
oraciones_tokenizadas_limpias

23


[['una',
  'bogotana',
  'el',
  'pasillo',
  'señora',
  'hermosa',
  'niña',
  'es',
  'como',
  'un',
  'lento',
  'rosado',
  'vals',
  'vea',
  'usted',
  'como',
  'aquellos',
  'dos',
  'enamorados',
  'pueden',
  'llevar',
  'el',
  'compas',
  'en',
  'medio',
  'de',
  'la',
  'mas',
  'ardiente',
  'conversacion',
  'el',
  'dice',
  'que',
  'los',
  'lindos',
  'ojos',
  'de',
  'una',
  'mujer',
  'valen',
  'por',
  'todos',
  'los',
  'astros',
  'los',
  'lindos',
  'labios',
  'por',
  'todas',
  'las',
  'rosas',
  'como',
  'ella',
  'quiere',
  'demostrar',
  'lo',
  'contrario',
  'le',
  'mira',
  'con',
  'los',
  'bellisimos',
  'ojos',
  'suyos',
  'le',
  'sonrie',
  'con',
  'sus',
  'inefables',
  'labios',
  'que',
  'son',
  'en',
  'un',
  'todo',
  'iguales',
  'aquellos',
  'con',
  'que',
  'la',
  'señorita',
  'de',
  'abril',
  'dio',
  'el',
  'primer',
  'beso',
  'al',
  'caballero',
  'de',
  'mayo',
  'el',
  'pasillo',
  'señora',
  'hermosa'

In [39]:
# tokenizacion en palabras
corpus = []
corpus_oraciones = []
for parrafo in parrafos.values():
  print(parrafo)
  try:
    corpus_oraciones.append( nltk.sent_tokenize(parrafo)[0].split() )
  except:
    print(parrafo)
  oraciones = [palabra for oracion in nltk.sent_tokenize(parrafo) for palabra in oracion.split()]
  corpus += oraciones
  print(corpus)

a una bogotana
['a', 'una', 'bogotana']
el pasillo señora hermosa niña es como un lento y rosado vals veausted como aquellos dos enamorados pueden llevar el compas en medio dela mas ardiente conversacion el dice que los lindos ojos de una mujervalen por todos los astros y los lindos labios por todas las rosascomo ella quiere demostrar lo contrario le mira con los bellisimos ojossuyos le sonrie con sus inefables labios que son en un todo iguales aaquellos con que la señorita de abril dio el primer beso al caballero demayo el pasillo señora hermosa niña es como un lento y rosado vals
['a', 'una', 'bogotana', 'el', 'pasillo', 'señora', 'hermosa', 'niña', 'es', 'como', 'un', 'lento', 'y', 'rosado', 'vals', 'veausted', 'como', 'aquellos', 'dos', 'enamorados', 'pueden', 'llevar', 'el', 'compas', 'en', 'medio', 'dela', 'mas', 'ardiente', 'conversacion', 'el', 'dice', 'que', 'los', 'lindos', 'ojos', 'de', 'una', 'mujervalen', 'por', 'todos', 'los', 'astros', 'y', 'los', 'lindos', 'labios', 'po

In [50]:
# Cantidad de tokens en este escrito
cantidad_tokens = Counter([token for oracion in oraciones_tokenizadas_limpias for token in oracion])

print(f'Cantidad de tokens {len(cantidad_tokens)}')
print(f'Populares {cantidad_tokens.most_common(10)}')

Cantidad de tokens 240
Populares [('de', 20), ('que', 20), ('la', 17), ('el', 15), ('un', 12), ('en', 10), ('una', 8), ('señora', 8), ('los', 8), ('como', 7)]


In [ ]:
# Importamos las bibliotecas para transformers
import faiss
import torch
import gensim
from gensim.models import Word2Vec, FastText

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

## Word2Vect

In [54]:
# instanciamos el modelo
# usando las oraciones
word2vec_model = Word2Vec(
    sentences=oraciones_tokenizadas_limpias,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg = 1,
    epochs = 10
    )


In [35]:
modelo.wv.key_to_index

{'que': 0,
 'de': 1,
 'la': 2,
 'y': 3,
 'el': 4,
 'un': 5,
 'en': 6,
 'a': 7,
 'los': 8,
 'una': 9,
 'señora': 10,
 'se': 11,
 'las': 12,
 'como': 13,
 'es': 14,
 'al': 15,
 'niña': 16,
 'hermosa': 17,
 'usted': 18,
 'por': 19,
 'vals': 20,
 'rosado': 21,
 'lento': 22,
 'pasillo': 23,
 'mi': 24,
 'abajo': 25,
 'con': 26,
 'lo': 27,
 'su': 28,
 'feliz': 29,
 'adelfa': 30,
 'rosa': 31,
 'lirio': 32,
 'heridas': 33,
 'esta': 34,
 'no': 35,
 'del': 36,
 'alla': 37,
 'tequendama': 38,
 'si': 39,
 'sus': 40,
 'le': 41,
 'quiere': 42,
 'ella': 43,
 'labios': 44,
 'ojos': 45,
 'lindos': 46,
 'dos': 47,
 'rosadovals': 48,
 'susiris': 49,
 'temblar': 50,
 'hacen': 51,
 'palmas': 52,
 'junto': 53,
 'rosas': 54,
 'sobre': 55,
 'quealla': 56,
 'espumas': 57,
 'locas': 58,
 'arrojela': 59,
 'corpiño': 60,
 'lleva': 61,
 'flor': 62,
 'memoriala': 63,
 'deshoje': 64,
 'gigantesco': 65,
 'llegue': 66,
 'favorcuando': 67,
 'dulce': 68,
 'hacerme': 69,
 'pero': 70,
 'paloma': 71,
 'adios': 72,
 'golondr

In [36]:
modelo.wv.get_vector('bogotana')

array([ 4.7800872e-03,  1.6491247e-03,  6.5619480e-03, -7.3351501e-03,
       -4.1776467e-03, -1.0647351e-03,  9.4348413e-04,  1.4716843e-03,
       -9.6747819e-03, -9.6018417e-03,  1.1629763e-03,  3.0833662e-03,
        9.4500845e-03, -8.6673386e-03, -2.9569401e-03, -6.4791758e-03,
        4.9088094e-03,  9.9152746e-03, -3.5016790e-03,  8.5163563e-03,
        4.4885203e-03, -6.1349734e-03,  3.3560139e-03, -2.2330986e-04,
       -9.0800300e-03,  3.8394523e-03, -1.2410154e-03,  1.1565689e-03,
        7.3449397e-03, -4.7672754e-03, -5.0801914e-03, -4.2522196e-03,
        1.9619712e-03,  4.1892510e-03, -9.8854555e-03, -4.5233676e-03,
        4.5045814e-03, -7.2386856e-03,  9.0266792e-03,  1.7205253e-03,
        5.8455620e-04, -2.6387835e-04, -3.3619541e-03, -2.0867137e-03,
       -3.0367225e-03,  6.2026726e-03,  8.5176574e-03, -5.8407439e-03,
       -7.3603122e-03,  5.4165027e-03, -8.2315272e-03,  6.3141081e-03,
        6.7286640e-03, -6.5014497e-03, -6.0122800e-03, -6.1175693e-03,
      

In [37]:
modelo.wv.most_similar('bogotana')

[('beso', 0.24641267955303192),
 ('quieren', 0.22161854803562164),
 ('si', 0.20873500406742096),
 ('bien', 0.203412264585495),
 ('veausted', 0.20333893597126007),
 ('lacteaa', 0.18955852091312408),
 ('ramo', 0.1828019618988037),
 ('ese', 0.1807553768157959),
 ('esta', 0.17622432112693787),
 ('señora', 0.17312461137771606)]

# Tarea

Tarea en 3 etapas
1. Poner nb en repo para poder cargarlo desde colab
1. Cargar cada uno de los archivos de texto (novelas y cuentos)
    1.1 Hacer limpieza de texto para quitar ya sea los asteriscos * * * o las anotaciones entre brackets [] y lo que consideres
2. Separar los textos en oraciones ajenas en un objeto en memoria (salvar las oraciones en disco)
3. Entrenar w2v con este dataset. Hacer pruebas de similaridad.
    3.1 Si se ven bien salvar los vectores con FAISS